# 01 — Setup and Data Inspection

Sanity-check the UD treebanks and both parsers on a handful of sentences.

In [ ]:
# === Kaggle / Colab setup (skip if running locally) ===
import os, subprocess, sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB  = "google.colab" in sys.modules

if IS_KAGGLE or IS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "../requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_trf"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "ru_core_news_lg"], check=True)
    subprocess.run([sys.executable, "../scripts/download_data.py"],
                   check=True)
    print("Setup done.")
else:
    print("Local env detected — skipping cloud setup.")


In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from src.data import load_sentences

EN = load_sentences(Path("../data/en_ewt_test.conllu"))
RU = load_sentences(Path("../data/ru_syntagrus_test.conllu"))

print(f"English-EWT test:       {len(EN)} sentences, {sum(len(s.tokens) for s in EN)} tokens")
print(f"Russian-SynTagRus test: {len(RU)} sentences, {sum(len(s.tokens) for s in RU)} tokens")

In [ ]:
import pandas as pd

df = pd.DataFrame(
    [{"lang": "en", "len": len(s.tokens)} for s in EN] +
    [{"lang": "ru", "len": len(s.tokens)} for s in RU]
)
df.groupby("lang")["len"].describe()

In [ ]:
import matplotlib.pyplot as plt
from src.plotting import apply_poster_style
apply_poster_style()

fig, ax = plt.subplots(figsize=(8, 4))
for lang, sents, color in [("en", EN, "steelblue"), ("ru", RU, "coral")]:
    lengths = [len(s.tokens) for s in sents]
    ax.hist(lengths, bins=40, alpha=0.6, label=lang, color=color)
ax.set_xlabel("Sentence length (tokens)")
ax.set_ylabel("Count")
ax.set_title("Sentence length distribution")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from src.parsers import SpacyParser, StanzaParser

sample_en = [EN[0].tokens]
spacy_en = SpacyParser("en_core_web_trf")
stanza_en = StanzaParser("en")

print("=== Sample sentence ===")
print("Tokens:", EN[0].tokens)
print()
print("GOLD   heads:", EN[0].heads)
print("GOLD   deprels:", EN[0].deprels)
print()
spacy_res = spacy_en.parse(sample_en)[0]
print("spaCy  heads:", spacy_res.heads)
print("spaCy  deprels:", spacy_res.deprels)
print()
stanza_res = stanza_en.parse(sample_en)[0]
print("Stanza heads:", stanza_res.heads)
print("Stanza deprels:", stanza_res.deprels)

## What we learned
- English-EWT has ~2K sentences; Russian-SynTagRus has ~6K
- Russian sentences tend to be longer on average
- Both parsers produce output in 1-indexed head format
- Ready to proceed to accuracy benchmarks (notebook 02)